# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding #2 — "The Content Performance Curve" (page 7)

**Claim:** health score follows a lifecycle — rising through the first 90 days, peaking at 61-90 days, decaying through 271-365 days, then partially recovering at 365+ if refreshed.

**Where does the label come from?** Health score at each age bucket is computed from a single cross-sectional snapshot: many *different* pages, each measured once, grouped by how old they happen to be right now. There is no per-page time series — the same `content_id` is not tracked from age 30 to age 300.

**Does the validation design support the claim?** Not fully. A lifecycle claim ("content ages, then decays") is a claim about a trajectory over time for a given piece of content. What's actually shown is a claim about a *population difference*: pages published a year ago look different, on average, from pages published last month. Those pages may differ in ways that have nothing to do with age itself — earlier cohorts might target different topics, face different competition, or have been produced under an older (weaker) content process. Without tracking the same pages over time, "content decays as it ages" and "older cohorts happen to be weaker for unrelated reasons" are both consistent with this chart. The paper does apply this kind of scrutiny elsewhere (e.g. flagging survivor bias in the tiny 365+×361+ cell) — it would be worth the same caveat here, since this chart anchors the paper's central refresh narrative.

---

### Finding #4 — "The Freshness Multiplier" (page 9)

**Claim:** refreshing 365+ day content produces a 3.2x health boost and 57x more impressions — described as "one of the strongest measured levers available."

**Where does the label come from?** The "refreshed" label is not randomly assigned. Some old pages get refreshed and some don't, and that choice is made by a person (at FlyRank or the client) — presumably someone who picked pages that already looked worth the effort: residual backlinks, a topic still in demand, a page that had ranked well before. The comparison group ("similar old pages, not refreshed") wasn't chosen the same way.

**Does the validation design support the claim?** Not for a causal "refresh → 57x impressions" story. This looks like **selection on the outcome dressed up as a treatment effect** — closer to the reverse-causality trap in the leakage taxonomy (the feature/decision is entangled with the outcome it's supposed to explain) than to a clean lever. To support the causal claim, you'd want to check whether refreshed pages already differed from never-refreshed pages *before* the refresh happened (prior impressions, prior backlink profile, prior position) — if refreshed pages were already trending better beforehand, the 57x is partly (or mostly) picking the winners, not making them.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["avg_position"] = df["avg_position"].replace(0, np.nan)

# Same drop list as w05_model.ipynb: label-derived columns, ID columns (grouping only,
# never features), and the *_last_30d trio that correlates 1.000 with trend_pct
# (window overlaps the label -- see the leakage audit in section 3 for the re-check).
LABEL_DERIVED = {"trend_direction", "trend_pct", "is_declining_label"}
ID_COLS = {"content_id", "client_id"}
LEAKY_WINDOW_COLS = {"impressions_last_30d", "clicks_last_30d", "sessions_last_30d"}
DROP_COLS = LABEL_DERIVED | ID_COLS | LEAKY_WINDOW_COLS

NUMERIC_FEATURES = [c for c in df.select_dtypes(include="number").columns if c not in DROP_COLS]
CATEGORICAL_FEATURES = [c for c in df.select_dtypes(include="object").columns if c not in DROP_COLS]


def make_pipeline() -> Pipeline:
    numeric_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocess = ColumnTransformer([
        ("num", numeric_pipe, NUMERIC_FEATURES),
        ("cat", categorical_pipe, CATEGORICAL_FEATURES),
    ])
    return Pipeline([("preprocess", preprocess), ("logreg", LogisticRegression(max_iter=1000, random_state=42))])


def precision_at_k(labels: pd.Series, k: int) -> float:
    return labels.head(k).mean()


def fit_and_score(train_idx, test_idx, split_name: str) -> list[dict]:
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    X_train = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
    y_train = train_df["is_declining_label"]
    X_test = test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
    y_test = test_df["is_declining_label"]

    pipe = make_pipeline()
    pipe.fit(X_train, y_train)
    test_df = test_df.copy()
    test_df["model_score"] = pipe.predict_proba(X_test)[:, 1]
    ranked = test_df.sort_values("model_score", ascending=False).reset_index(drop=True)

    n_clients = test_df["client_id"].nunique()
    rows = []
    for k in (10, 50):
        rows.append({
            "split": split_name,
            "k": k,
            "precision@k": precision_at_k(ranked["is_declining_label"], k),
            "test_base_rate": y_test.mean(),
            "test_clients": n_clients,
        })
    return rows


# BEFORE -- plain random split. Same test_size and random_state as the honest split
# below, but rows are shuffled with no regard for client_id, so a client's rows can
# land in both train and test.
random_splitter = ShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
random_train_idx, random_test_idx = next(random_splitter.split(df))
results = fit_and_score(random_train_idx, random_test_idx, "BEFORE: random split")

# AFTER -- grouped by client_id (from w05_model.ipynb). No client's rows appear in
# both train and test, so the model can't "cheat" by memorizing a client's quirks.
group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
group_train_idx, group_test_idx = next(group_splitter.split(df, df["is_declining_label"], groups=df["client_id"]))
results += fit_and_score(group_train_idx, group_test_idx, "AFTER: grouped by client")

comparison = pd.DataFrame(results)
print(comparison.to_string(index=False))


                   split  k  precision@k  test_base_rate  test_clients
    BEFORE: random split 10         0.80        0.544556            32
    BEFORE: random split 50         0.84        0.544556            32
AFTER: grouped by client 10         0.80        0.559442            10
AFTER: grouped by client 50         0.74        0.559442            10


**Before/after, same LR pipeline, same features, same `test_size=0.3` / `random_state=42`:**

| split | k | precision@k |
|---|---|---|
| BEFORE: random split | 10 | 0.80 |
| BEFORE: random split | 50 | **0.84** |
| AFTER: grouped by client | 10 | 0.80 |
| AFTER: grouped by client | 50 | **0.74** |

At k=50 the random split beats the grouped split by 10 points (0.84 vs 0.74). At k=10 the two
happen to tie — with only 10 rows in play, that's not evidence the splits are equivalent, just
a coin-flip-sized sample.

**Why the gap exists.** In the random split, rows are shuffled without regard to `client_id`,
so all 32 clients still show up in *both* train and test — just with fewer rows each in test.
That gives the model a shortcut: it can partly learn "this specific client's pages tend to
decline at roughly rate X" from seeing that client elsewhere in training, rather than learning
a signal that would transfer to a client it has never seen. In the grouped split, the 10 test
clients are held out entirely — the model can only succeed by learning something about content
characteristics (staleness, position, traffic mix) that generalizes across clients, which is
the harder and more honest bar.

**Which number to trust.** The grouped-split precision@50 (0.74) is the one that reflects how
this model would actually behave in the situation it's meant for — ranking decline risk for a
client's content, very possibly a client the model wasn't trained on. The random-split number
(0.84) overstates that by roughly 10 points, entirely from client memorization, not from any
real gain in the model's understanding of decline. This also matches the leakage skill's
general expectation: a random split is rarely honest when rows share a repeating entity, and
the size of the gap here (10 points at k=50) is itself a measure of how much memorization was
happening.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
from sklearn.metrics import roc_auc_score

# 1. Correlation scan -- does any remaining feature look suspiciously close to the label
# or to trend_pct (the number the label is derived from)? A sibling of a label-derived
# column would show up here even if it isn't literally named "trend_*".
corr_with_label = (
    df[NUMERIC_FEATURES + ["is_declining_label"]]
    .corr()["is_declining_label"]
    .drop("is_declining_label")
    .sort_values(key=abs, ascending=False)
)
print("Numeric features most correlated with is_declining_label:")
print(corr_with_label.head(8).to_string())

print("\nNo product-flag / composite-score columns exist in this starter CSV "
      "(health_score, optimization flags are FlyRank-paper concepts, not in the "
      "44-column dataset) -- confirmed against the data dictionary, so that leakage "
      "vector doesn't apply here by construction.")

# 2. Verify the harness itself catches leakage: deliberately add back
# impressions_last_30d (already known, from w05, to correlate 1.000 with trend_pct)
# and confirm precision@K spikes toward 1.0 -- if it doesn't, the test setup is broken,
# not the data.
LEAK_TEST_FEATURES = NUMERIC_FEATURES + ["impressions_last_30d"]


def fit_and_score_features(train_idx, test_idx, numeric_cols) -> dict:
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)
    cols = numeric_cols + CATEGORICAL_FEATURES

    preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORICAL_FEATURES),
    ])
    pipe = Pipeline([("preprocess", preprocess), ("logreg", LogisticRegression(max_iter=1000, random_state=42))])
    pipe.fit(train_df[cols], train_df["is_declining_label"])
    scores = pipe.predict_proba(test_df[cols])[:, 1]
    ranked = test_df.assign(model_score=scores).sort_values("model_score", ascending=False).reset_index(drop=True)
    return {
        "auc": roc_auc_score(test_df["is_declining_label"], scores),
        "precision@10": precision_at_k(ranked["is_declining_label"], 10),
        "precision@50": precision_at_k(ranked["is_declining_label"], 50),
    }


honest = fit_and_score_features(group_train_idx, group_test_idx, NUMERIC_FEATURES)
with_leak = fit_and_score_features(group_train_idx, group_test_idx, LEAK_TEST_FEATURES)

print("\nLeakage-harness check (grouped split, same train/test as section 2):")
print(f"  without impressions_last_30d (final feature set): {honest}")
print(f"  WITH impressions_last_30d added back:              {with_leak}")


Numeric features most correlated with is_declining_label:
days_with_impressions     0.190055
content_age_days         -0.163882
age_tier_order           -0.156142
word_count                0.090157
days_since_last_update    0.081383
avg_position             -0.081304
char_count                0.072188
ctr                      -0.061911

No product-flag / composite-score columns exist in this starter CSV (health_score, optimization flags are FlyRank-paper concepts, not in the 44-column dataset) -- confirmed against the data dictionary, so that leakage vector doesn't apply here by construction.



Leakage-harness check (grouped split, same train/test as section 2):
  without impressions_last_30d (final feature set): {'auc': 0.6180589179365189, 'precision@10': np.float64(0.8), 'precision@50': np.float64(0.74)}
  WITH impressions_last_30d added back:              {'auc': 0.8747795346790832, 'precision@10': np.float64(1.0), 'precision@50': np.float64(1.0)}


**Checklist, against the [hunting-leakage-and-validating](../../skills/hunting-leakage-and-validating/SKILL.md) taxonomy:**

- **Timeline:** every remaining feature is a `*_90d` or `*_prev_30d` aggregate, or static
  content metadata (word_count, content_type, age). None of them are computed from a window
  that overlaps `trend_pct`'s comparison window (last 30d vs prev 30d) — that check is what
  dropped the `*_last_30d` trio back in Week 5.
- **No label-derived or sibling columns:** the correlation scan above shows nothing close to
  1.0 in the final feature set — the strongest is `days_with_impressions` at 0.19, weak enough
  to be a real (if modest) signal rather than a reconstruction of the label.
- **No product flags:** confirmed against the data dictionary — this starter CSV has no
  `health_score` / optimization-flag columns to accidentally include.
- **Split grouped:** yes, by `client_id` (section 2).
- **Base rate printed:** yes, next to every precision@k in section 2 (`test_base_rate`).
- **Harness verification:** deliberately adding `impressions_last_30d` back sends AUC from
  0.618 → 0.875 and precision@10/@50 both straight to **1.0** — the skill's own "suspiciously
  perfect" tell, and proof the test setup is sensitive enough to catch real leakage rather than
  silently passing anything. The Week-5 catch was correct: that column really was reconstructing
  the label almost exactly.
- **Out-of-fold:** all metrics above are computed on the held-out grouped test set only, never
  on training rows.

**Conclusion:** the final feature set clears the checklist. The honest number to carry forward
is the grouped-split, leak-free precision@50 of **0.74** (AUC 0.618) from section 2 — modest,
but real.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**The claim, as originally written (Week-5 notebook, section 4):**

> "LR still clears the base rate comfortably (0.74 vs 0.56 at k=50) and edges the rule at k=50."

**Why this overclaims.** "Edges the rule at k=50" describes a single number's worth of
difference — 0.74 (LR) vs 0.72 (rule) — from **one** train/test split, one `random_state`, no
repeated runs, no confidence interval. This week's own before/after in section 2 showed
precision@50 moving from 0.74 to 0.84 on the *same model* just by changing the split — an
8x-larger swing than the 0.02 gap the original sentence leans on. A 0.02 edge from a single
split is well within the noise this notebook already demonstrated exists, so "edges" implies a
settled comparison that the evidence doesn't support.

**Rewritten, safe language:**

> On this grouped, client-held-out split, logistic regression's precision@50 was **observed**
> at 0.74 versus the rule baseline's 0.72 — a small, directional difference that is not larger
> than the variation seen when only the train/test split changes (0.74 → 0.84 across splits in
> section 2). This is decision-support evidence that LR is at least competitive with the rule,
> not measured proof that it outperforms it. A repeated-split or cross-validated comparison
> would be needed before recommending LR over the rule on this margin alone.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.